# APDS TP03 — Gemma 3 (Colab)

## Setup

Select a GPU runtime (L4 recommended). For W&B, add a Colab secret named `WANDB_API_KEY` and enable notebook access. Set `WANDB_PROJECT = None` to skip tracking.

In [ ]:
%pip install -q "unsloth==2026.11.2" "transformers==5.17.0" "trl==1.15.0" "sacrebleu==2.5.1" wandb

In [ ]:
from pathlib import Path
import os
import subprocess

checkout = Path("/content/year-05-itc-apds-tp03")
if not checkout.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "tp-03-colab",
         "https://github.com/menghong-heng/year-05-itc-apds.git", str(checkout)],
        check=True,
    )
os.chdir(checkout / "tp3")

## Experiment

In [ ]:
from scripts.training import (
    build_trainer, evaluate_answers, generate_answers, load_model,
    prepare_run, save_run, tokenize_splits, track_run,
)

EXPERIMENT = "khmerqa_2epochs"
WANDB_PROJECT = "apds-tp03"

In [ ]:
if WANDB_PROJECT:
    from google.colab import userdata

    try:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise RuntimeError("Add WANDB_API_KEY in Colab Secrets and grant notebook access.") from None

## Data and baseline

In [ ]:
run = prepare_run(EXPERIMENT)
model, tokenizer = load_model()
dataset = tokenize_splits(run.splits, tokenizer)
test_rows = run.splits["test"]
before = generate_answers(model, tokenizer, test_rows)

## Train and compare

In [ ]:
with track_run(run, WANDB_PROJECT) as tracking:
    trainer = build_trainer(run, model, tokenizer, dataset, track=tracking is not None)
    stats = trainer.train()
    after = generate_answers(trainer.model, tokenizer, test_rows)
    results, scores = evaluate_answers(test_rows, before, after)
    save_run(run, trainer, tokenizer, stats, results, scores, tracking)

## Download results

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

archive = run.path.with_suffix(".zip")
artifacts = [*run.path.glob("*.json"), *run.path.glob("*.csv"),
             *run.path.glob("*.png"), *run.path.joinpath("adapter").rglob("*")]
with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for artifact in artifacts:
        if artifact.is_file():
            bundle.write(artifact, artifact.relative_to(run.path))
files.download(str(archive))